# make a pydantic class specifying datatypes and stuff 
# instantiate it 

mind that you have to unpack the dictionary

In [15]:
from pydantic import BaseModel

class Patient(BaseModel):
    name : str 
    age : int 

patient_dict = {'name':'Ananthan' , 'age':20}

patient1 = Patient(**patient_dict)

def insert_patient(function_patient : Patient):
    print(function_patient.name)
    print(function_patient.age)
    print("Inserted")

updated_patient_1 = {'name':'Ananthan' , 'age':30}

def update_patient(update_patient : Patient):
    print(update_patient.name)
    print(update_patient.age)
    print("updated")


update_patient_1 = Patient(**updated_patient_1)
update_patient(update_patient_1)

insert_patient(patient1)



Ananthan
30
updated
Ananthan
20
Inserted


# no matter how many fields are there , if we write code like this all the fields are marked as required . 

In [20]:
from pydantic import BaseModel 
from typing import List , Dict 

class Batient(BaseModel):
    name : str 
    age : int 
    marriage : bool
    allergies : List[str]
    contact_info : Dict[str , str]

patient_dict_1 = {'name':'Ananthan' , 'age':49 , 'marriage':True , 'allergies':['pollen' , 'dust'] , 'contact_info':{'email':'things@gmail.com' , 'phone':'9874345678754'}}

patient_1 = Batient(**patient_dict_1) 

def insert_patient(pat:Patient):
    print(pat.name)
    print(pat.contact_info)
    print(pat.allergies)
    print("inserted")

insert_patient(patient_1)

Ananthan
{'email': 'things@gmail.com', 'phone': '9874345678754'}
['pollen', 'dust']
inserted


we can create optional datatype from the optional class from the typing library , we have to set a default valuse if that is the case .
you can put the deafult values without the optional class

In [27]:
from typing import Optional 
from pydantic import EmailStr , AnyUrl

class  P(BaseModel):
    name : str 
    age : int 
    email : EmailStr 
    linkdin : AnyUrl
    marriage : int = False
    allergies : Optional[List[str]] = None 
    contact_info : Optinal[Dict[str , str]] = None

we can perform data validation in pydantic . they provide some custom datatype to do so - Emailstr and Anyurl

# custom data validation using field function 

In [29]:
from pydantic import Annotated , Field 

class p(BaseModel):
    name : Annotated[str , Field(max_length = 50 , title = "name of the patient" , description = 'give the name of the patient in less than 50 characters') , example=['ananthan' , 'savio']]
    email : EmailStr
    linkdin : AnyUrl 
    age : int = Field(gt=0 , lt=99)
    weight : float = Field(gt=0)
    married :Annotated[bool , Field(default = none , description = 'whether married or not ?')]
    allergies : Annotated[Dict[str , str] , Field(default = none , description = 'list all the allergies')]


SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (3266462568.py, line 4)

# you can use fiel validator to transform the contents inside of an instance and also for data validation
# you can you the model validator to validate two or more dependenet fields . if not validated that model will not be creted 

In [19]:
from pydantic import BaseModel , EmailStr ,field_validator ,model_validator
from typing import Dict , List 

class Patient(BaseModel):
    name : str 
    age : int 
    email : EmailStr
    weight : float 
    married : bool
    contact_info : Dict[str , str]
    allergies : List[str]

    @field_validator('email' , mode = 'after')
    @classmethod
    def data_validate_email(cls , value):
        domains = ['hdfc.com' , 'icici.com']
        given_domain = value.split('@')[-1]
        if given_domain not in domains :
            raise ValueError("Incorrect emploeye")
        return value 

    @field_validator('name' , mode = 'before')
    @classmethod
    def convert_name_into_upper_case(cls , value):
        return value.upper()

    @model_validator(mode='before')
    @classmethod
    def check_for_age_and_contact_details(cls , self):
        if self.age > 60 and 'emergency' not in self.contact_info:
            raise ValueError ("Provide emergency contact number for patients above 60")
        return self 

patient_info = {'name':'ananthan' , 'age':7 , 'email':'abc@icici.com' , 'weight':77.77 , 'married':False , 'contact_info':{'phone':'12345432'} , 'allergies':['dust' , 'pollen']}

p = Patient(**patient_info)

def update_patient (function_patient:Patient):
    print(function_patient.name)
    print(function_patient.email)
    print("Patient updated")

update_patient(p)


AttributeError: 'dict' object has no attribute 'age'

In [22]:
from pydantic import BaseModel , EmailStr ,field_validator ,model_validator ,computed_field
from typing import Dict , List 

class Patient(BaseModel):
    name : str 
    age : int 
    email : EmailStr
    weight : float 
    height : float
    married : bool
    contact_info : Dict[str , str]
    allergies : List[str]

    @computed_field
    @property
    def compute_bmi(self) -> float:
        bmi = self.weight/(self.height**2)
        return bmi

patient_info = {'name':'ananthan' , 'age':7 , 'email':'abc@icici.com' , 'weight':77.77 ,'height':97.8, 'married':False , 'contact_info':{'phone':'12345432'} , 'allergies':['dust' , 'pollen']}

p = Patient(**patient_info)

def update_patient (function_patient:Patient):
    print(function_patient.name)
    print(function_patient.email)
    print(function_patient.compute_bmi)
    print("Patient updated")

update_patient(p)

ananthan
abc@icici.com
0.00813082079783875
Patient updated


In [25]:
from pydantic import BaseModel 

class Address(BaseModel):
    house_name : str 
    ward : str 
    district : str 

class Patient(BaseModel):
    name : str 
    age : int 
    gender : str 
    marriage : bool 
    address : Address

address_dict = {'house_name':'chakkarat' , 'ward':'Muncipal office' , 'district':'Alappuzha'}

address1 = Address(**address_dict)

patient_dict = {'name':'John Doe' , 'age':30 , 'gender':'Male' , 'marriage':True , 'address':address1}

patient1 = Patient(**patient_dict)

def update_second_patient(function_patient : Patient):
    print(function_patient.name)
    print(function_patient.address.ward)
    print("updation done")

update_second_patient(patient1)

John Doe
Muncipal office
updation done


# we can export pydantic models as python dictionary 

In [ ]:
dictioanary = patient1.model_dump(include = ['name' , 'gender'])

In [29]:
print(dictioanary)

{'name': 'John Doe', 'gender': 'Male'}


In [30]:
dictioanary = patient1.model_dump(exclude = ['name' , 'gender'])

In [31]:
print(dictioanary)

{'age': 30, 'marriage': True, 'address': {'house_name': 'chakkarat', 'ward': 'Muncipal office', 'district': 'Alappuzha'}}


In [32]:
print(type(dictioanary))

<class 'dict'>
